# PIT fundamentals event study — the anti-look-ahead differentiator (H.89.6)

**[Open in Colab](https://colab.research.google.com/github/BlueWaterCorp/RiskModels_API/blob/main/sdk/notebooks/pit_fundamentals_event_study.ipynb)** · **[Get API key](https://riskmodels.app/get-key)**

Most fundamentals APIs key on `period_end_date` ("give me the Q2 numbers") and silently return whatever the **current, restated** value is — including data that was not actually knowable on any date near the quarter end. `GET /fundamentals/{ticker}`'s `as_of` parameter is the differentiator: a row is visible **iff `filed_date <= as_of`**, never "latest". This notebook proves the mechanism by walking `as_of` across a real filing boundary and watching a quarter's data appear exactly once knowable — not before.

**Scope note:** the store's `eps_actual` / `eps_estimate` (and therefore a true EPS-surprise event study) are held back pending SEC-promotion or counsel clearance (`FUNDAMENTALS_HELD_BACK_FIELDS`, H.69) — this SDK never reconstructs a held-back field client-side (see `CLAUDE.md`). So this walks the **same PIT mechanism** using the fields that *are* shipped (`roe_ttm`, `cost_of_equity`, `wacc`) — the identical `as_of`/`filed_date` gate a real surprise event study will use once those fields clear review.


### Colab only (skip locally)

Run once to install the SDK; local users should `pip install riskmodels-py` in their venv.


In [1]:
import sys

try:
    import google.colab  # noqa: F401
    _COLAB = True
except ImportError:
    _COLAB = False

if _COLAB:
    import subprocess

    _deps = ["python-dotenv"]
    _pypi = "riskmodels-py>=0.3.4"
    _git = (
        "riskmodels-py @ git+https://github.com/BlueWaterCorp/RiskModels_API.git"
        "@main#subdirectory=sdk"
    )
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _pypi, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py from PyPI (+ python-dotenv).")
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _git, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print(
            "Colab: installed riskmodels-py from GitHub main "
            "(PyPI may not list this version yet)."
        )
else:
    print("Local: use your environment (pip install riskmodels-py python-dotenv).")


Local: use your environment (pip install riskmodels-py python-dotenv).


## 1. Connect — `RiskModelsClient.from_env()`

Loads **`RISKMODELS_API_KEY`** from shell env, `.env` / `.env.local` (when `python-dotenv` is installed), or Colab Secrets.


In [2]:
import os

from IPython.display import display

from riskmodels import RiskModelsClient
from riskmodels.client import DEFAULT_BASE_URL
from riskmodels.notebook import load_notebook_dotenv

load_notebook_dotenv()
print("RISKMODELS_BASE_URL =", os.environ.get("RISKMODELS_BASE_URL", DEFAULT_BASE_URL))
client = RiskModelsClient.from_env()


RISKMODELS_BASE_URL = https://riskmodels.app/api


## 2. Find a recent filing boundary

Pull enough history to see `period_end_date` / `filed_date` pairs, then pick the most recent one as the event boundary.


In [3]:
TICKER = "AAPL"

history = client.get_fundamentals(TICKER, periods=8, as_dataframe=True)
display(history[["period_end_date", "filed_date", "filed_date_source", "roe_ttm"]])

event = history.iloc[-1]
event_period_end = event["period_end_date"]
event_filed = event["filed_date"]
print(f"Event quarter: period_end={event_period_end}, filed_date={event_filed}")


,period_end_date,filed_date,filed_date_source,roe_ttm
0,2024-06-30,2024-08-02,exact,1.471503
1,2024-09-30,2024-11-01,exact,1.378714
2,2024-12-31,2025-01-31,exact,1.453460
3,2025-03-31,2025-05-02,exact,1.513055
4,2025-06-30,2025-08-01,exact,1.549229
5,2025-09-30,2025-10-31,exact,1.640469
6,2025-12-31,2026-01-30,exact,1.599422
7,2026-03-31,2026-05-01,exact,1.466892


Event quarter: period_end=2026-03-31, filed_date=2026-05-01


## 3. Naive `period_end` cutoff vs. true PIT `filed_date` cutoff

A backtest that (incorrectly) uses the quarter's `period_end_date` as its knowledge cutoff would already "see" a quarter's data — but nobody actually knew those numbers until the filing landed, ~1-2 months later. Querying `as_of=period_end_date` should **not** surface the event quarter; only `as_of=filed_date` should.


In [4]:
naive = client.get_fundamentals(TICKER, as_of=event_period_end, periods=8)
pit = client.get_fundamentals(TICKER, as_of=event_filed, periods=8)

naive_has_event = any(r["period_end_date"] == event_period_end for r in naive["rows"])
pit_has_event = any(r["period_end_date"] == event_period_end for r in pit["rows"])

print(f"as_of=period_end_date ({event_period_end}): event quarter visible = {naive_has_event}")
print(f"as_of=filed_date      ({event_filed}):      event quarter visible = {pit_has_event}")
assert not naive_has_event and pit_has_event, (
    "Expected the event quarter to be invisible at as_of=period_end and visible "
    "at as_of=filed_date — if this fails, filed_date_source may be 'approx' "
    "(period_end + 45d), which can coincide with the as_of used above."
)


as_of=period_end_date (2026-03-31): event quarter visible = False
as_of=filed_date      (2026-05-01):      event quarter visible = True


## 4. Step function across the filing date

Walk `as_of` day-by-day around `filed_date` and show `roe_ttm` for the event quarter appearing at exactly one point — never gradually, never early.


In [5]:
from datetime import date, timedelta

filed = date.fromisoformat(event_filed)
rows = []
for offset in range(-5, 6):
    as_of = (filed + timedelta(days=offset)).isoformat()
    body = client.get_fundamentals(TICKER, as_of=as_of, periods=8)
    visible_row = next(
        (r for r in body["rows"] if r["period_end_date"] == event_period_end), None
    )
    rows.append(
        {
            "as_of": as_of,
            "days_from_filing": offset,
            "event_quarter_visible": visible_row is not None,
            "roe_ttm": visible_row["roe_ttm"] if visible_row else None,
        }
    )

import pandas as pd

step = pd.DataFrame(rows)
display(step)


,as_of,days_from_filing,event_quarter_visible,roe_ttm
0,2026-04-26,-5,False,NaN
1,2026-04-27,-4,False,NaN
2,2026-04-28,-3,False,NaN
3,2026-04-29,-2,False,NaN
4,2026-04-30,-1,False,NaN
5,2026-05-01,0,True,1.466892
6,2026-05-02,1,True,1.466892
7,2026-05-03,2,True,1.466892
8,2026-05-04,3,True,1.466892
9,2026-05-05,4,True,1.466892


## Reading the table

`event_quarter_visible` should flip from `False` to `True` exactly at `days_from_filing == 0` and stay `True` afterward — a hard step, not a gradual reveal. That step is the whole point: any backtest, screen, or agent that queries this endpoint with a historical `as_of` gets exactly the information that existed on that date, no more.

## Next steps

- Once `eps_actual` / `eps_estimate` clear H.69 counsel review, a true surprise event study reuses this identical `as_of` walk — swap `roe_ttm` for the surprise field and the mechanism is unchanged.
- **Quality-overlay screen:** [`quality_overlay_screen.ipynb`](./quality_overlay_screen.ipynb)
- **Pre-earnings hedge timing:** [`pre_earnings_hedge_timing.ipynb`](./pre_earnings_hedge_timing.ipynb)
